# 06 Tongue Scenario Demo

This notebook walks through a crop-substitution scenario: five alfalfa
fields in the Tongue basin switched to corn.  It shows the scenario
specification, the NDVI changes, and how the model's ET and irrigation
outputs respond.

Full workflow context:
[Tongue Scenario Analysis](../workflows/tongue-scenario-analysis.md)

Artifacts read here: scenario TOML, scenario container, hindcast
container, diagnostic summary CSV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from swimrs.container import SwimContainer
from swim_mtdnrc.scenarios.scenario_spec import ScenarioSpec

HINDCAST_PATH = "/nas/swim/examples/tongue_ensemble/data/tongue_hindcast.swim"
SCENARIO_PATH = "/nas/swim/examples/tongue_ensemble/data/tongue_test_corn.swim"
TOML_PATH = "/nas/swim/examples/tongue_ensemble/data/test_corn_scenario.toml"
SUMMARY_CSV = "/nas/swim/examples/tongue_ensemble/reports/test_corn/scenario_summary.csv"

HINDCAST_RUN = "hindcast_core"
SCENARIO_RUN = "test_corn_5fields"

spec = ScenarioSpec.from_toml(TOML_PATH)
fids = [s.fid for s in spec.substitutions]

hindcast = SwimContainer.open(HINDCAST_PATH, mode="r")
scenario = SwimContainer.open(SCENARIO_PATH, mode="r")

print(f"Scenario: {spec.name}")
print(f"Substituted fields: {fids} -> {spec.substitutions[0].target_crop}")
print(f"Hindcast run: {HINDCAST_RUN}  |  Scenario run: {SCENARIO_RUN}")

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
summary

In [ ]:
# Before/after NDVI for one substituted field (2015-2020)
fid = fids[1]  # 1930
ndvi_h = hindcast.query.dataframe(
    "derived/merged_ndvi/irr", fields=[fid],
    start_date="2015-01-01", end_date="2020-12-31"
)[fid]
ndvi_s = scenario.query.dataframe(
    "derived/merged_ndvi/irr", fields=[fid],
    start_date="2015-01-01", end_date="2020-12-31"
)[fid]

fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(ndvi_h.index, ndvi_h.values, color="steelblue", lw=0.7,
        label="hindcast (alfalfa)", alpha=0.8)
ax.plot(ndvi_s.index, ndvi_s.values, color="orangered", lw=0.7,
        label="scenario (corn)", alpha=0.8)
ax.set_ylabel("NDVI")
ax.set_title(f"FID {fid} — NDVI before and after substitution")
ax.legend(fontsize=8)
ax.set_ylim(0, 0.9)
plt.show()

In [ ]:
# Annual ET comparison for all substituted fields
rows = []
for fid in fids:
    eta_h = hindcast.run_dataframe(HINDCAST_RUN, fid, variables=["eta"])["eta"]
    eta_s = scenario.run_dataframe(SCENARIO_RUN, fid, variables=["eta"])["eta"]
    h_annual = eta_h.resample("YE").sum().mean()
    s_annual = eta_s.resample("YE").sum().mean()
    rows.append({
        "FID": fid,
        "Hindcast ET (mm/yr)": round(h_annual, 1),
        "Scenario ET (mm/yr)": round(s_annual, 1),
        "Change (mm/yr)": round(s_annual - h_annual, 1),
        "Change (%)": round(100 * (s_annual - h_annual) / h_annual, 1),
    })

et_df = pd.DataFrame(rows).set_index("FID")
et_df

In [ ]:
# Monthly mean ET for FID 1930 — shows seasonal timing shift
fid = fids[1]
eta_h = hindcast.run_dataframe(HINDCAST_RUN, fid, variables=["eta"])["eta"]
eta_s = scenario.run_dataframe(SCENARIO_RUN, fid, variables=["eta"])["eta"]

monthly_h = eta_h.groupby(eta_h.index.month).mean()
monthly_s = eta_s.groupby(eta_s.index.month).mean()

months = np.arange(1, 13)
labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(months, monthly_h.values, "o-", color="steelblue",
        label="hindcast (alfalfa)", markersize=5)
ax.plot(months, monthly_s.values, "o-", color="orangered",
        label="scenario (corn)", markersize=5)
ax.set_xticks(months)
ax.set_xticklabels(labels)
ax.set_ylabel("Mean daily ET (mm/day)")
ax.set_title(f"FID {fid} — Monthly mean ET")
ax.legend(fontsize=8)
plt.show()

In [ ]:
hindcast.close()
scenario.close()